# 6-2 requires_grad와 Tensor gradient — 심화

직접 작성한 코드와 저장된 실행 결과를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
# encoder 출력의 detach가 head 학습은 남기면서 encoder 연결만 끊는 상황을 동일 batch에서 재현합니다.
import copy
import torch
import torch.nn as nn

torch.manual_seed(0)
base_encoder = nn.Linear(2, 2, bias=False)
base_head = nn.Linear(2, 1, bias=False)
x = torch.tensor([[1.0, -1.0]])

def run(detach_features):
    encoder = copy.deepcopy(base_encoder)
    head = copy.deepcopy(base_head)
    features = encoder(x)
    # 전체 미세조정에서는 이 분리가 없어야 합니다.
    if detach_features:
        features = features.detach()
    loss = head(features).pow(2).mean()
    loss.backward()
    return encoder.weight.grad is None, head.weight.grad is None

detached = run(True)
fixed = run(False)
# detach 제거 전후 두 모듈의 grad None 여부를 나란히 출력해 전체 loss 단절과 encoder 단절을 구분합니다.
print(f"detached_encoder_grad_none={detached[0]}")
print(f"detached_head_grad_none={detached[1]}")
print(f"fixed_encoder_grad_none={fixed[0]}")
print(f"fixed_head_grad_none={fixed[1]}")

detached_encoder_grad_none=True
detached_head_grad_none=False
fixed_encoder_grad_none=False
fixed_head_grad_none=False


In [2]:
# 검증 가능 정답 코드
# encoder의 requires_grad를 끄고 optimizer에는 trainable parameter만 넘겨 고정 정책을 코드로 일치시킵니다.
import torch
import torch.nn as nn

torch.manual_seed(3)
model = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2))

# 첫 Linear 전체를 고정합니다. weight만 고정하고 bias를 놓치면 계약이 깨집니다.
for p in model[0].parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
optimizer = torch.optim.SGD((p for p in model.parameters() if p.requires_grad), lr=0.1)

first_before = model[0].weight.detach().clone()
head_before = model[2].weight.detach().clone()
x = torch.tensor([[1.0, 0.0, -1.0, 2.0], [0.5, 1.0, 0.0, -1.0]])
loss = model(x).pow(2).mean()
optimizer.zero_grad()
loss.backward()
optimizer.step()

# 한 step 전후 state를 비교해 encoder 15개는 그대로이고 head 8개만 실제로 바뀌는지 확인합니다.
print(f"trainable_params={trainable}")
print(f"frozen_params={frozen}")
print(f"encoder_unchanged={torch.equal(first_before, model[0].weight)}")
print(f"head_changed={not torch.equal(head_before, model[2].weight)}")

trainable_params=8
frozen_params=15
encoder_unchanged=True
head_changed=True


In [3]:
# 검증 가능 정답 코드
# 후보 로그의 encoder/head gradient 신호를 전체 미세조정이 아닌 고정-encoder 운영 계약으로 평가합니다.
runs = {
    "A": {"encoder_grad": False, "head_grad": True, "trainable": 8, "valid_loss": 0.61},
    "B": {"encoder_grad": True, "head_grad": True, "trainable": 23, "valid_loss": 0.58},
}

# 계약을 먼저 필터로 적용하고, metric은 통과한 후보 안에서만 비교합니다.
eligible = [name for name, r in runs.items() if not r["encoder_grad"] and r["head_grad"]]
approved = min(eligible, key=lambda name: runs[name]["valid_loss"])
# 적격 후보를 고른 뒤 gradient 신호만 믿지 않고 encoder weight 전후 비교를 재검사 항목으로 남깁니다.
print(f"eligible={eligible}")
print(f"approved={approved}")
print("recheck=compare encoder weights before and after step")

eligible=['A']
approved=A
recheck=compare encoder weights before and after step
